In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Running outside Colab: files are expected under /src
pass


In [3]:
#All relevant imports for the notebook
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import requests
import datetime
from datetime import datetime
import sqlite3
import numpy as np
from scipy import stats
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from thefuzz import process
import geopandas as gpd
from shapely.geometry import Point


In [4]:
import pandas as pd
from pathlib import Path

# Use project-relative paths (works when cwd is project root; /src/ fails on Windows)
SRC = Path("src")
def _path(name):
    p = SRC / name
    if not p.exists() and "Arrests" in name and " (1)" not in name:
        alt = SRC / "NYPD_Arrests_Data__Historic_ (1).csv"
        return str(alt) if alt.exists() else str(p)
    return str(p)

airbnb_nyc_df = pd.read_csv(_path("AB_NYC_2019.csv"))
nypd_shooting_df = pd.read_csv(_path("NYPD_Shooting_Incident_Data__Historic_.csv"))
ny_tree_census_df = pd.read_csv(_path("new_york_tree_census_2015.csv"))
total_population_df = pd.read_csv(_path("Total Population.csv"), header=4)
arrest_data_df = pd.read_csv(_path("NYPD_Arrests_Data__Historic_.csv"))
nta_pop_df = pd.read_csv(_path("New_York_City_Population_By_Neighborhood_Tabulation_Areas.csv"))
nyc_gdf = gpd.read_file(_path("nycgeo.json"))
"""
display(ny_tree_census_df.head())
display(nypd_shooting_df.head())
display(airbnb_nyc_df.head())
display(Total_Population_df.head())
"""

'\ndisplay(ny_tree_census_df.head())\ndisplay(nypd_shooting_df.head())\ndisplay(airbnb_nyc_df.head())\ndisplay(Total_Population_df.head())\n'

#**Most impactful factors affecting Airbnb pricing in New York City**

**Hypothesis: Greenery, Reviews, and Crime Rate affect Airbnb prices**

As Airbnb grows in popularity across the world, getting an accurate estimate of the price of Airbnbs will help new Airbnb owners to better price their houses. We will explore the relationship between various factors of the Airbnb and its price. We believe that crime rate, greenery around the borough and the reviews of the Airbnb will have the greatest impact on its price.

#**Data source identification and exploration(Prepare)**

In order to estimate the price of Airbnbs, we have extracted various data points such as room type, number of reviews, availability, boroughs, along with the three factors in our hypothesis. We have selected three datasets which are connected by the 5 types of boroughs in New York City.

List of datasets:
- [New York City Airbnb Open Data (2019) ](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data)
- [NYPD Arrest Data (2006-2019) ](https://www.kaggle.com/datasets/thaddeussegura/nypd-arrest-data-20062019)
- [New York City Shooting Incident Dataset](https://www.kaggle.com/datasets/thaddeussegura/new-york-city-shooting-dataset)
- [Tree Census in New York City](https://www.kaggle.com/datasets/nycparks/tree-census)
- [Total Population in New York City](https://data.cccnewyork.org/data/map/97/total-population#97/a/3/147/132/a/a)

In [5]:
def map_coordinates_to_neighbourhood(df):
    import geopandas as gpd
    import pandas as pd

    temp_df = df.copy()

    # Detect latitude / longitude columns case-insensitively.
    lower_to_original = {c.lower(): c for c in temp_df.columns}
    lat_col = lower_to_original.get('latitude')
    lon_col = lower_to_original.get('longitude')

    if lat_col is None or lon_col is None:
        raise ValueError("Input dataframe must contain latitude/longitude columns.")

    # Coerce to numeric and keep only valid coordinate rows for spatial join.
    temp_df[lat_col] = pd.to_numeric(temp_df[lat_col], errors='coerce')
    temp_df[lon_col] = pd.to_numeric(temp_df[lon_col], errors='coerce')
    valid_mask = (
        temp_df[lat_col].between(-90, 90)
        & temp_df[lon_col].between(-180, 180)
        & temp_df[lat_col].notna()
        & temp_df[lon_col].notna()
    )

    valid_df = temp_df.loc[valid_mask].copy()

    # Initialize neighbourhood to preserve original row count/order.
    temp_df['neighbourhood'] = pd.NA
    if valid_df.empty:
        return temp_df

    gdf = gpd.GeoDataFrame(
        valid_df,
        geometry=gpd.points_from_xy(valid_df[lon_col], valid_df[lat_col]),
        crs='EPSG:4326',
    )

    nyc_gdf_local = nyc_gdf[['geometry', 'NTAName']].copy()
    if nyc_gdf_local.crs is None:
        nyc_gdf_local = nyc_gdf_local.set_crs('EPSG:4326')
    elif nyc_gdf_local.crs.to_string() != 'EPSG:4326':
        nyc_gdf_local = nyc_gdf_local.to_crs('EPSG:4326')

    joined = gpd.sjoin(gdf, nyc_gdf_local, how='left', predicate='within')

    mapped = fuzzy_map_neighbourhood(joined, 'NTAName')
    temp_df.loc[mapped.index, 'neighbourhood'] = mapped['neighbourhood']

    return temp_df


def fuzzy_map_neighbourhood(df, target_column):
    from thefuzz import process

    temp_df = df.copy()
    airbnb_unique_hoods = [
        n for n in airbnb_nyc_df['neighbourhood'].unique()
        if n != 'Co-op City'
    ]

    source_hoods = temp_df[target_column].dropna().unique().tolist()

    fuzz_map = {}
    for hood in source_hoods:
        match, score = process.extractOne(hood, airbnb_unique_hoods)
        fuzz_map[hood] = match if score >= 75 else None

    temp_df['neighbourhood'] = temp_df[target_column].map(fuzz_map)
    return temp_df

###DS1: New York City Airbnb Open Data(2019)

Description: The NYC Airbnb Open Data dataset, downloaded on 28/01/2026, contains detailed listing activity and metrics including geographical locations, prices, and room types across New York City. Specifically, it provides neighbourhood_group (the five boroughs) and neighbourhood data, which offer an ideal level of granularity for analyzing market density and local pricing trends. The dataset also includes the last_review date, allowing for temporal analysis of listing popularity and guest activity. The following code imports the data into a DataFrame and displays the first few rows to illustrate the structure of the rental market data.

In [6]:
#We first take a look at the firsst 5 rows of the dataset
airbnb_nyc_df.head()
airbnb_nyc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

In [7]:
description_df = airbnb_nyc_df[['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']].describe().round(1)
description_df = description_df.transpose()
description_df = description_df.rename(columns={
    'count': 'Number of Listings',
    'mean': 'Average Value',
    'std': 'Standard Deviation',
    'min': 'Minimum Value',
    '25%': '25th Percentile',
    '50%': 'Median Value',
    '75%': '75th Percentile',
    'max': 'Maximum Value'
})
display(description_df)

,Number of Listings,Average Value,Standard Deviation,Minimum Value,25th Percentile,Median Value,75th Percentile,Maximum Value
price,48895.0,152.7,240.2,0.0,69.0,106.0,175.0,10000.0
minimum_nights,48895.0,7.0,20.5,1.0,1.0,3.0,5.0,1250.0
number_of_reviews,48895.0,23.3,44.6,0.0,1.0,5.0,24.0,629.0
reviews_per_month,38843.0,1.4,1.7,0.0,0.2,0.7,2.0,58.5
calculated_host_listings_count,48895.0,7.1,33.0,1.0,1.0,1.0,2.0,327.0
availability_365,48895.0,112.8,131.6,0.0,0.0,45.0,227.0,365.0


In [8]:
# Check for total number of null values for each column
#
empty_records = airbnb_nyc_df.isnull().sum()
empty_records[empty_records > 0]

name                    16
host_name               21
last_review          10052
reviews_per_month    10052
dtype: int64

In [9]:
airbnb_nyc_df['neighbourhood'] = airbnb_nyc_df['neighbourhood'].apply(lambda x: x.split(',')[0].split('-')[0].strip())
# Exclude 'Co-op City' from the neighbourhood column
filtered_airbnb_df = airbnb_nyc_df[airbnb_nyc_df['neighbourhood'] != 'Co-op City']
average_price_by_neighbourhood = filtered_airbnb_df.groupby('neighbourhood')['price'].mean().reset_index()
display(average_price_by_neighbourhood.head(20))

,neighbourhood,price
0,Allerton,87.595238
1,Arden Heights,67.250000
2,Arrochar,115.000000
3,Arverne,171.779221
4,Astoria,117.187778
5,Bath Beach,81.764706
6,Battery Park City,367.557143
7,Bay Ridge,144.432624
8,Bay Terrace,132.125000
9,Baychester,75.428571


In [10]:
airbnb_nyc_df['neighbourhood'].nunique()

220

###DS2: New York City Shooting Incident(2019)

Description: The Kaggle dataset, focusing on historic shooting incidents, provides a detailed log of public safety events including the exact date, time, and geographic coordinates of each occurrence in 2019. More precisely, borough-level data, which offers the granularity that we require. This allows for the creation of "safety proximity" features that can significantly influence localized pricing models. The following code imports the data into a DataFrame and displays the first few rows, offering an initial look at the incident reports and their spatial attributes.

In [11]:
nypd_shooting_df.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,PRECINCT,JURISDICTION_CODE,LOCATION_DESC,STATISTICAL_MURDER_FLAG,PERP_AGE_GROUP,PERP_SEX,PERP_RACE,VIC_AGE_GROUP,VIC_SEX,VIC_RACE,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,Lon_Lat
0,74146165,08/14/2010,3:11:00,QUEENS,113,0.0,NaN,False,NaN,NaN,NaN,25-44,M,BLACK,1046573,183057,40.668915,-73.775341,POINT (-73.77534099699994 40.66891477200004)
1,66928846,10/17/2009,18:03:00,BROOKLYN,67,0.0,NaN,True,NaN,NaN,NaN,45-64,M,BLACK,1003313,176413,40.650877,-73.931302,POINT (-73.93130224699998 40.65087729100002)
2,29114164,05/18/2007,23:00:00,BROOKLYN,75,0.0,NaN,False,NaN,NaN,NaN,25-44,M,BLACK,1016292,176228,40.650332,-73.884529,POINT (-73.884529479 40.65033205800006)
3,85180336,06/09/2012,17:15:00,BROOKLYN,81,0.0,NaN,False,NaN,NaN,NaN,25-44,M,BLACK,1005597,188673,40.684523,-73.923032,POINT (-73.92303235699995 40.68452304300007)
4,73405770,06/27/2010,4:14:00,BRONX,47,0.0,NaN,False,NaN,NaN,NaN,25-44,M,BLACK,1023551,263366,40.889474,-73.857860,POINT (-73.85786021699995 40.88947350500007)


In [12]:
nypd_shooting_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21626 entries, 0 to 21625
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   INCIDENT_KEY             21626 non-null  int64  
 1   OCCUR_DATE               21626 non-null  object 
 2   OCCUR_TIME               21626 non-null  object 
 3   BORO                     21626 non-null  object 
 4   PRECINCT                 21626 non-null  int64  
 5   JURISDICTION_CODE        21624 non-null  float64
 6   LOCATION_DESC            9265 non-null   object 
 7   STATISTICAL_MURDER_FLAG  21626 non-null  bool   
 8   PERP_AGE_GROUP           14321 non-null  object 
 9   PERP_SEX                 14355 non-null  object 
 10  PERP_RACE                14355 non-null  object 
 11  VIC_AGE_GROUP            21626 non-null  object 
 12  VIC_SEX                  21626 non-null  object 
 13  VIC_RACE                 21626 non-null  object 
 14  X_COORD_CD            

In [13]:
total_population_df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Location,TimeFrame,DataFormat,Data,Fips
1,Battery Park/Tribeca,2005,Number,51150.45963,101
2,Greenwich Village,2005,Number,75508.54037,102
3,Lower East Side,2005,Number,145556,103
4,Chelsea/Clinton,2005,Number,88754.39184,104


In [14]:
nypd_shooting_df = map_coordinates_to_neighbourhood(nypd_shooting_df)

In [15]:
# Calculate value counts for each matched neighborhood
incident_counts = nypd_shooting_df['neighbourhood'].value_counts().reset_index()
incident_counts.columns = ['neighbourhood', 'incident_count']

# Display the top 20 neighborhoods by incident count
display(incident_counts.head(20))

,neighbourhood,incident_count
0,Harlem,1447
1,East New York,1120
2,Crown Heights,1079
3,Brownsville,864
4,Bedford,764
5,Mott Haven,687
6,Stuyvesant Town,662
7,Flatbush,555
8,Tremont,531
9,Bushwick,518


In [16]:
# 1. Prepare unique Airbnb neighborhoods (excluding Co-op City)
airbnb_unique_hoods = [n for n in airbnb_nyc_df['neighbourhood'].unique() if n != 'Co-op City']

# 2. Extract NTA population reference (Year 2010)
nta_2010 = nta_pop_df[nta_pop_df['Year'] == 2010].groupby('NTA Name')['Population'].sum().reset_index()
nta_names = nta_2010['NTA Name'].tolist()

# 3. Perform Fuzzy Matching (75% threshold)
mapping_results = []
for hood in airbnb_unique_hoods:
    match, score = process.extractOne(hood, nta_names)
    pop_val = nta_2010.loc[nta_2010['NTA Name'] == match, 'Population'].values[0] if score >= 75 else None
    mapping_results.append({'Airbnb Neighbourhood': hood, 'Population (2010 Census)': pop_val})

# 4. Create and display the final mapping table
airbnb_pop_map_df = pd.DataFrame(mapping_results).sort_values('Airbnb Neighbourhood')

print(f'Created population mapping for {len(airbnb_pop_map_df)} neighborhoods.')
display(airbnb_pop_map_df.head(20))

Created population mapping for 220 neighborhoods.


,Airbnb Neighbourhood,Population (2010 Census)
69,Allerton,28903.0
214,Arden Heights,25238.0
86,Arrochar,16079.0
106,Arverne,36885.0
56,Astoria,78793.0
197,Bath Beach,29931.0
101,Battery Park City,39699.0
90,Bay Ridge,79371.0
140,Bay Terrace,21751.0
187,Baychester,34517.0


In [17]:
# 1. Merge incident counts with the population mapping
# airbnb_pop_map_df contains the population per Airbnb neighbourhood
# incident_counts contains total shootings per airbnb_matched_neighbourhood

neighbourhood_stats = incident_counts.merge(
    airbnb_pop_map_df,
    left_on='neighbourhood',
    right_on='Airbnb Neighbourhood',
    how='inner'
)

# 2. Calculate shootings per 10,000 people
# We ensure we don't divide by zero and only include valid population data
neighbourhood_stats = neighbourhood_stats[neighbourhood_stats['Population (2010 Census)'] > 0]

neighbourhood_stats['shootings_per_10000'] = (
    neighbourhood_stats['incident_count'] / neighbourhood_stats['Population (2010 Census)']
) * 10000

# 3. Sort by the new metric
neighbourhood_stats = neighbourhood_stats.sort_values(by='shootings_per_10000', ascending=False)

# Display the top results
print("Neighborhoods with the highest shooting density (per 10,000 residents):")
display(neighbourhood_stats[['neighbourhood', 'incident_count', 'Population (2010 Census)', 'shootings_per_10000']].head(20))

Neighborhoods with the highest shooting density (per 10,000 residents):


,neighbourhood,incident_count,Population (2010 Census),shootings_per_10000
6,Stuyvesant Town,662,21049.0,314.504252
0,Harlem,1447,75282.0,192.210621
15,Prospect,353,19849.0,177.842712
5,Mott Haven,687,39214.0,175.192533
16,Lighthouse Hill,352,22887.0,153.799100
3,Brownsville,864,58300.0,148.198971
8,Tremont,531,43423.0,122.285425
1,East New York,1120,91958.0,121.794732
4,Bedford,764,70713.0,108.042368
2,Crown Heights,1079,103169.0,104.585680


###DS3 and DS4: NYPD Arrest Data and Total Population

Description for NYPD Arrest Data: The Kaggle dataset, encompassing historic NYPD arrest records from 2006 to 2024, provides a comprehensive record of every arrest effected in New York City. This dataset includes highly specific geospatial coordinates and other markers such as date and time, which are vital for a long-term analysis of neighborhood safety evolution. More precisely, it contains the borough and precinct, which are at the exact level of granularity required. For our Airbnb smart pricing project, this allows us to track how the safety profile of a specific borough and its market value—has shifted over nearly two decades. The following code imports the data into a DataFrame and displays the first few rows, highlighting the offense categories and location descriptors used in our predictive model.

Description for Total Population Data: The provided dataset, sourced from the U.S. Census Bureau, contains historic Total Population figures for New York City across various years, including our target year of 2019. The data is organized by Location, providing the borough-level granularity necessary to analyze market density and demand. This allows for a more nuanced understanding of Airbnb pricing by enabling us to calculate "per capita" metrics for crime or amenity availability. The following code imports the data into a DataFrame and displays the first few rows, showing the population counts and the specific community districts or neighborhoods covered.

In [18]:
arrest_data_df.head()

,ARREST_KEY,ARREST_DATE,PD_CD,PD_DESC,KY_CD,OFNS_DESC,LAW_CODE,LAW_CAT_CD,ARREST_BORO,ARREST_PRECINCT,JURISDICTION_CODE,AGE_GROUP,PERP_SEX,PERP_RACE,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,Lon_Lat
0,10837169,04/02/2006,NaN,NaN,NaN,NaN,NaN,NaN,Q,101,0.0,NaN,M,BLACK,1051775.0,159727.0,40.604841,-73.756823,POINT (-73.75682250899997 40.604840985000074)
1,189797412,11/09/2018,475.0,NaN,NaN,NaN,PL 1651601,M,M,28,1.0,25-44,F,WHITE HISPANIC,997374.0,234664.0,40.810773,-73.952592,POINT (-73.95259158999993 40.81077276700007)
2,189992103,11/14/2018,475.0,NaN,NaN,NaN,PL 1651601,M,K,75,1.0,18-24,M,BLACK,1021568.0,185710.0,40.676337,-73.865464,POINT (-73.86546353999995 40.676337363000066)
3,189714430,11/07/2018,117.0,RECKLESS ENDANGERMENT 1,126.0,MISCELLANEOUS PENAL LAW,PL 1202500,F,M,26,0.0,45-64,M,WHITE,993685.0,233346.0,40.807160,-73.965920,POINT (-73.96591978699998 40.80715993100006)
4,190067816,11/15/2018,586.0,NaN,NaN,NaN,PL 230341A,F,M,5,0.0,18-24,M,BLACK,984946.0,200203.0,40.716196,-73.997491,POINT (-73.99749074599998 40.716195914000025)


In [19]:
arrest_data_df.describe()

,ARREST_KEY,PD_CD,KY_CD,ARREST_PRECINCT,JURISDICTION_CODE,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
count,5.012956e+06,5.012695e+06,5.003927e+06,5.012956e+06,5.012946e+06,5.012955e+06,5.012955e+06,5.012955e+06,5.012955e+06
mean,9.579149e+07,5.117004e+02,3.016261e+02,6.062439e+01,1.304714e+00,1.005357e+06,2.149659e+05,4.075649e+01,-7.392378e+01
std,5.222361e+07,2.633719e+02,1.817349e+02,3.430862e+01,9.427788e+00,1.999599e+04,1.648721e+05,4.456872e-01,7.216800e-02
min,9.926901e+06,0.000000e+00,1.010000e+02,1.000000e+00,0.000000e+00,9.133570e+05,1.211310e+05,4.049891e+01,-7.425494e+01
25%,5.931854e+07,2.930000e+02,1.260000e+02,3.300000e+01,0.000000e+00,9.933700e+05,1.868860e+05,4.067957e+01,-7.396707e+01
50%,8.345894e+07,5.110000e+02,3.410000e+02,6.000000e+01,0.000000e+00,1.004890e+06,2.094910e+05,4.074166e+01,-7.392549e+01
75%,1.435467e+08,7.500000e+02,3.480000e+02,8.400000e+01,0.000000e+00,1.015835e+06,2.366140e+05,4.081609e+01,-7.388586e+01
max,2.068936e+08,9.970000e+02,9.950000e+02,1.230000e+02,9.700000e+01,1.067302e+06,8.202360e+06,6.208307e+01,-7.368178e+01


In [20]:
arrest_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5012956 entries, 0 to 5012955
Data columns (total 19 columns):
 #   Column             Dtype  
---  ------             -----  
 0   ARREST_KEY         int64  
 1   ARREST_DATE        object 
 2   PD_CD              float64
 3   PD_DESC            object 
 4   KY_CD              float64
 5   OFNS_DESC          object 
 6   LAW_CODE           object 
 7   LAW_CAT_CD         object 
 8   ARREST_BORO        object 
 9   ARREST_PRECINCT    int64  
 10  JURISDICTION_CODE  float64
 11  AGE_GROUP          object 
 12  PERP_SEX           object 
 13  PERP_RACE          object 
 14  X_COORD_CD         float64
 15  Y_COORD_CD         float64
 16  Latitude           float64
 17  Longitude          float64
 18  Lon_Lat            object 
dtypes: float64(7), int64(2), object(10)
memory usage: 726.7+ MB


In [21]:
arrest_data_df = map_coordinates_to_neighbourhood(arrest_data_df)

print(arrest_data_df)

         ARREST_KEY ARREST_DATE  PD_CD                         PD_DESC  KY_CD  \
0          10837169  04/02/2006    NaN                             NaN    NaN   
1         189797412  11/09/2018  475.0                             NaN    NaN   
2         189992103  11/14/2018  475.0                             NaN    NaN   
3         189714430  11/07/2018  117.0         RECKLESS ENDANGERMENT 1  126.0   
4         190067816  11/15/2018  586.0                             NaN    NaN   
...             ...         ...    ...                             ...    ...   
5012951   206858026  12/27/2019  478.0  THEFT OF SERVICES, UNCLASSIFIE  343.0   
5012952   205304150  11/18/2019  922.0  TRAFFIC,UNCLASSIFIED MISDEMEAN  348.0   
5012953   205307020  11/18/2019  922.0  TRAFFIC,UNCLASSIFIED MISDEMEAN  348.0   
5012954   206301725  12/10/2019  779.0  PUBLIC ADMINISTRATION,UNCLASSI  126.0   
5012955   206235317  12/09/2019  109.0        ASSAULT 2,1,UNCLASSIFIED  106.0   

                           

In [22]:
total_population_df[5:].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3081 entries, 5 to 3085
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  3081 non-null   object
 1   Unnamed: 1  3081 non-null   object
 2   Unnamed: 2  3081 non-null   object
 3   Unnamed: 3  3081 non-null   object
 4   Unnamed: 4  3081 non-null   object
dtypes: object(5)
memory usage: 120.5+ KB


In [23]:
total_population_df = pd.read_csv(str(Path("src") / "Total Population.csv"), header=4)
total_population_df.columns = total_population_df.iloc[0]
total_population_df = total_population_df[1:].reset_index(drop=True)
total_population_df = total_population_df.rename(columns={
    'Location': 'Borough',
    'TimeFrame': 'Year',
    'Data': 'Population'
})
total_population_df['Year'] = pd.to_numeric(total_population_df['Year'])
total_population_df['Population'] = pd.to_numeric(total_population_df['Population'])
target_boroughs = ['Manhattan', 'Queens', 'Brooklyn', 'Bronx', 'Staten Island']
target_years = range(2015, 2020)
filtered_population = total_population_df[
    (total_population_df['Borough'].isin(target_boroughs)) &
    (total_population_df['Year'].isin(target_years))
]
display(filtered_population[['Borough', 'Year', 'Population']])

,Borough,Year,Population
1655,Bronx,2015,1455444.0
1656,Brooklyn,2015,2636735.0
1657,Manhattan,2015,1644518.0
1658,Queens,2015,2339150.0
1659,Staten Island,2015,474558.0
1946,Bronx,2016,1455720.0
1947,Brooklyn,2016,2629150.0
1948,Manhattan,2016,1643734.0
1949,Queens,2016,2333054.0
1950,Staten Island,2016,476015.0


In [24]:
# Arrest density per 10,000 residents, by year
# NYPD arrests dataset uses these coordinate columns: 'Latitude' and 'Longitude'.
arrest_cols_needed = ['ARREST_DATE', 'Latitude', 'Longitude']
arrest_input = arrest_data_df[arrest_cols_needed].copy()

print("Mapping arrests to neighbourhood using Latitude/Longitude...")
arrest_mapped = map_coordinates_to_neighbourhood(arrest_input)
arrest_mapped = arrest_mapped.dropna(subset=['neighbourhood']).copy()

# 1. Parse ARREST_DATE to extract year
arrest_mapped['year'] = pd.to_datetime(arrest_mapped['ARREST_DATE'], format='%m/%d/%Y', errors='coerce').dt.year
arrest_mapped = arrest_mapped.dropna(subset=['year'])
arrest_mapped['year'] = arrest_mapped['year'].astype(int)

# 2. Count arrests per neighbourhood per year
arrest_counts_by_year = arrest_mapped.groupby(['neighbourhood', 'year']).size().reset_index(name='arrest_count')

# 3. Merge with population (2010 Census) for density calculation
arrest_density_by_year = arrest_counts_by_year.merge(
    airbnb_pop_map_df,
    left_on='neighbourhood',
    right_on='Airbnb Neighbourhood',
    how='inner'
)
arrest_density_by_year = arrest_density_by_year[arrest_density_by_year['Population (2010 Census)'] > 0]
arrest_density_by_year['arrests_per_10000'] = (
    arrest_density_by_year['arrest_count'] / arrest_density_by_year['Population (2010 Census)']
) * 10000

# 4. Display summary: top neighbourhoods by arrests_per_10000 (averaged across years)
arrest_summary = arrest_density_by_year.groupby('neighbourhood').agg(
    total_arrests=('arrest_count', 'sum'),
    mean_arrests_per_10000=('arrests_per_10000', 'mean'),
    years_covered=('year', 'nunique')
).reset_index().sort_values('mean_arrests_per_10000', ascending=False)

print("Arrest density by neighbourhood (mean arrests per 10,000 residents across years):")
display(arrest_summary.head(20))
print("\nArrest density by neighbourhood and year:")
display(arrest_density_by_year[['neighbourhood', 'year', 'arrest_count', 'Population (2010 Census)', 'arrests_per_10000']].head(25))

Mapping arrests to neighbourhood using Latitude/Longitude...


Arrest density by neighbourhood (mean arrests per 10,000 residents across years):


,neighbourhood,total_arrests,mean_arrests_per_10000,years_covered
88,Midtown,178694,4458.210668,14
67,Harlem,333063,3160.146421,14
92,Mott Haven,139682,2544.317263,14
120,Stuyvesant Town,61767,2096.027636,14
71,Hunts Point,69725,1830.744428,14
32,Clinton Hill,83064,1705.367152,14
132,Washington Heights,156474,1664.787042,14
74,Jamaica,120199,1597.299186,14
103,Prospect,43116,1551.571508,14
80,Lighthouse Hill,44856,1399.921353,14



Arrest density by neighbourhood and year:


,neighbourhood,year,arrest_count,Population (2010 Census),arrests_per_10000
0,Allerton,2006,492,28903.0,170.224544
1,Allerton,2007,523,28903.0,180.950074
2,Allerton,2008,498,28903.0,172.300453
3,Allerton,2009,612,28903.0,211.742726
4,Allerton,2010,644,28903.0,222.814241
5,Allerton,2011,631,28903.0,218.316438
6,Allerton,2012,595,28903.0,205.860983
7,Allerton,2013,815,28903.0,281.977649
8,Allerton,2014,725,28903.0,250.839013
9,Allerton,2015,644,28903.0,222.814241


#**DS 5 Tree Census**

Description: The 2015 NYC Street Tree Census dataset, a landmark "citizen science" project known as TreesCount!, provides a comprehensive digital inventory of 666,134 street trees mapped across all five boroughs. This dataset is organized by individual tree records and can be organised by borough level. More precisely, it contains qualitative assessments of tree health and stewardship, which serve as proxies for neighborhood maintenance and resident engagement at the exact level of granularity required to categorize listings by street-level appeal.For our smart pricing project, this allows us to track how the density of trees in a specific boroughcan affect property prices. The following code imports the data into a DataFrame and displays the first few rows, showing the diameter at breast height (tree_dbh) and environmental indicators used in our predictive model.

In [25]:
# 1. Apply fuzzy mapping to align tree census 'nta_name' with Airbnb neighborhoods
# Note: 'nta_name' is the target column in ny_tree_census_df
tree_census_mapped = fuzzy_map_neighbourhood(ny_tree_census_df, 'nta_name')

# 2. Exclude 'Co-op City' and records that didn't meet the 75% threshold
tree_census_mapped = tree_census_mapped[tree_census_mapped['neighbourhood'].notnull()]

# 3. Calculate total number of trees per matched neighborhood
tree_counts_per_neighborhood = tree_census_mapped.groupby('neighbourhood').size().reset_index(name='total_trees')

# 4. Sort and display the results
tree_counts_per_neighborhood = tree_counts_per_neighborhood.sort_values(by='total_trees', ascending=False)
display(tree_counts_per_neighborhood.head(20))

,neighbourhood,total_trees
74,Jamaica,15861
99,Park Slope,13128
51,Eltingville,12969
47,East New York,12598
56,Flushing,11459
67,Harlem,10790
65,Great Kills,10734
9,Bayside,9780
135,West Village,9548
143,Woodrow,9251


In [26]:
# 1. Load and project GeoJSON for accurate area calculation
if 'nyc_gdf' not in globals():
    nyc_gdf = gpd.read_file(str(Path("src") / "nycgeo.json"))

nyc_projected = nyc_gdf.to_crs(epsg=2263)
nyc_projected['area_sq_miles'] = nyc_projected['geometry'].area / (5280**2)

# 2. Extract area data
raw_area_df = nyc_projected[['NTAName', 'area_sq_miles']].copy()

# 3. Apply fuzzy mapping to align with Airbnb neighborhoods
# We use the previously defined fuzzy_map_neighbourhood function
mapped_area_df = fuzzy_map_neighbourhood(raw_area_df, 'NTAName')

# 4. Clean up: Remove records that didn't match and aggregate area
# (In case multiple NTAs map to the same Airbnb neighborhood)
neighborhood_area_df = mapped_area_df.dropna(subset=['neighbourhood'])
neighborhood_area_df = neighborhood_area_df.groupby('neighbourhood')['area_sq_miles'].sum().reset_index()

# 5. Display the results
print("Square Area mapped to Airbnb Neighborhoods:")
display(neighborhood_area_df.sort_values(by='area_sq_miles', ascending=False).head(20))

Square Area mapped to Airbnb Neighborhoods:


,neighbourhood,area_sq_miles
99,Park Slope,34.378008
95,New Springville,11.736441
52,Emerson Hill,6.632073
74,Jamaica,5.714650
125,Tottenville,5.220325
51,Eltingville,5.064445
47,East New York,4.893792
56,Flushing,3.807754
121,Sunnyside,3.674021
12,Belle Harbor,3.573575


In [27]:
# Tree density per neighbourhood (trees per square mile)
tree_density_df = tree_counts_per_neighborhood.merge(
    neighborhood_area_df,
    on='neighbourhood',
    how='inner'
)

# Avoid divide-by-zero
tree_density_df = tree_density_df[tree_density_df['area_sq_miles'] > 0]

tree_density_df['trees_per_sq_mile'] = (
    tree_density_df['total_trees'] / tree_density_df['area_sq_miles']
)

tree_density_df = tree_density_df.sort_values('trees_per_sq_mile', ascending=False)

print('Tree density by neighbourhood (trees per sq mile):')
display(tree_density_df[['neighbourhood', 'total_trees', 'area_sq_miles', 'trees_per_sq_mile']].head(20))

Tree density by neighbourhood (trees per sq mile):


,neighbourhood,total_trees,area_sq_miles,trees_per_sq_mile
57,Upper East Side,4673,0.719746,6492.566472
126,Brooklyn Heights,1767,0.358157,4933.586195
36,Upper West Side,5881,1.233211,4768.850541
142,Fordham,1060,0.226300,4684.051095
110,Windsor Terrace,2290,0.503683,4546.508783
141,Gramercy,1172,0.269959,4341.407016
61,Glendale,4509,1.076703,4187.783026
134,Prospect Heights,1535,0.367248,4179.737435
131,Longwood,1572,0.385464,4078.206320
130,East Village,1575,0.390902,4029.138080


In [28]:
# 1. Get the list of unique Airbnb neighborhoods
airbnb_unique_hoods = set([n for n in airbnb_nyc_df['neighbourhood'].unique() if n != 'Co-op City'])

# 2. Get the list of successfully mapped neighborhoods
mapped_hoods = set(neighborhood_area_df['neighbourhood'].unique())

# 3. Find the difference
unmapped_hoods = airbnb_unique_hoods - mapped_hoods

# 4. Display the results
print(f"Total unique Airbnb neighborhoods: {len(airbnb_unique_hoods)}")
print(f"Neighborhoods mapped to area data: {len(mapped_hoods)}")
print(f"Neighborhoods NOT included: {len(unmapped_hoods)}")

if len(unmapped_hoods) > 0:
    print("\nSample of neighborhoods not included:")
    print(list(unmapped_hoods)[:20])

Total unique Airbnb neighborhoods: 220
Neighborhoods mapped to area data: 145
Neighborhoods NOT included: 75

Sample of neighborhoods not included:
['Neponsit', 'Tribeca', 'North Riverdale', 'Mill Basin', 'Melrose', 'Little Neck', 'Jamaica Estates', 'Morris Heights', 'Rockaway Beach', 'Manhattan Beach', 'Riverdale', 'Mariners Harbor', 'Rosebank', 'Downtown Brooklyn', 'Schuylerville', "Prince's Bay", "Hell's Kitchen", 'Kips Bay', 'East Morrisania', 'Flatiron District']


In [29]:
ny_tree_census_df.head()

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,st_assem,st_senate,nta,nta_name,boro_ct,state,latitude,longitude,x_sp,y_sp
0,606945,305778,2016-06-28,10,0,OnCurb,Alive,Good,Fraxinus pennsylvanica,green ash,...,25,14,QN37,Kew Gardens Hills,4125700,New York,40.724339,-73.805180,1.038250e+06,203232.9417
1,160321,341273,2015-08-19,9,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,34,13,QN28,Jackson Heights,4030902,New York,40.756626,-73.894167,1.013571e+06,214953.6472
2,541347,325281,2015-12-30,7,0,OnCurb,Alive,Good,Pyrus calleryana,Callery pear,...,32,10,QN76,Baisley Park,4028800,New York,40.679777,-73.788463,1.042923e+06,187008.2671
3,613930,203822,2016-07-05,10,0,OnCurb,Alive,Good,Pyrus calleryana,Callery pear,...,46,22,BK31,Bay Ridge,3005000,New York,40.622743,-74.037543,9.738279e+05,166160.5847
4,18353,338911,2015-06-13,4,0,OnCurb,Alive,Good,Prunus virginiana,'Schubert' chokecherry,...,31,10,QN12,Hammels-Arverne-Edgemere,4095400,New York,40.596514,-73.797622,1.040452e+06,156667.5017


In [30]:
ny_tree_census_df['boroname'].value_counts()

boroname
Queens           250551
Brooklyn         177293
Staten Island    105318
Bronx             85203
Manhattan         65423
Name: count, dtype: int64

In [31]:
# 1. Prepare unique Airbnb neighborhoods (excluding Co-op City)
airbnb_unique_hoods = [n for n in airbnb_nyc_df['neighbourhood'].unique() if n != 'Co-op City']

# 2. Extract NTA population reference (Year 2010)
nta_2010 = nta_pop_df[nta_pop_df['Year'] == 2010].groupby('NTA Name')['Population'].sum().reset_index()
nta_names = nta_2010['NTA Name'].tolist()

# 3. Perform Fuzzy Matching (75% threshold)
mapping_results = []
for hood in airbnb_unique_hoods:
    match, score = process.extractOne(hood, nta_names)
    pop_val = nta_2010.loc[nta_2010['NTA Name'] == match, 'Population'].values[0] if score >= 75 else None
    mapping_results.append({'Airbnb Neighbourhood': hood, 'Population (2010 Census)': pop_val})

# 4. Create and display the final mapping table
airbnb_pop_map_df = pd.DataFrame(mapping_results).sort_values('Airbnb Neighbourhood')

print(f'Created population mapping for {len(airbnb_pop_map_df)} neighborhoods.')
display(airbnb_pop_map_df.head(20))

Created population mapping for 220 neighborhoods.


,Airbnb Neighbourhood,Population (2010 Census)
69,Allerton,28903.0
214,Arden Heights,25238.0
86,Arrochar,16079.0
106,Arverne,36885.0
56,Astoria,78793.0
197,Bath Beach,29931.0
101,Battery Park City,39699.0
90,Bay Ridge,79371.0
140,Bay Terrace,21751.0
187,Baychester,34517.0
